# Shakespeare Speech Similarity Analysis
This notebook analyzes the semantic similarity between consecutive speeches in the Folger Shakespeare corpus.
We use BERT-based cosine similarity scores stored in `output/speech_interactions_bert.csv`.

In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Add src to path to import metadata
sys.path.append('src')
from genre_analysis import PLAYS

sns.set_theme(style='whitegrid', palette='muted')

## 1. Data Loading and Preparation
We load the interaction data and join it with genre/year metadata from the original corpus definition.

In [9]:
df = pd.read_csv('output/speech_interactions_bert.csv')

# Create metadata dataframe
play_metadata = pd.DataFrame(PLAYS, columns=['path', 'title', 'genre', 'year'])

# Merge metadata
df = df.merge(play_metadata[['title', 'genre', 'year']], left_on='play', right_on='title', how='left')
df.drop(columns=['title'], inplace=True)

print(f'Loaded {len(df)} interactions across {df["play"].nunique()} plays.')
df.head()

Loaded 30258 interactions across 37 plays.


,play,scene,speaker1,speaker2,text1,text2,cosine_similarity,model,genre,year
0,The Comedy of Errors,A1.S1,Egeon_Err,Duke_Err,"Proceed, Solinus, to procure my fall,\nAnd by ...","Merchant of Syracusa, plead no more.\nI am not...",0.766615,bert,comedy,1594.0
1,The Comedy of Errors,A1.S1,Duke_Err,Egeon_Err,"Merchant of Syracusa, plead no more.\nI am not...","Yet this my comfort: when your words are done,...",0.783361,bert,comedy,1594.0
2,The Comedy of Errors,A1.S1,Egeon_Err,Duke_Err,"Yet this my comfort: when your words are done,...","Well, Syracusian, say in brief the cause\nWhy ...",0.793440,bert,comedy,1594.0
3,The Comedy of Errors,A1.S1,Duke_Err,Egeon_Err,"Well, Syracusian, say in brief the cause\nWhy ...",A heavier task could not have been imposed\nTh...,0.759901,bert,comedy,1594.0
4,The Comedy of Errors,A1.S1,Egeon_Err,Duke_Err,A heavier task could not have been imposed\nTh...,"Nay, forward, old man. Do not break off so,\nF...",0.749138,bert,comedy,1594.0


## 2. Global Distribution of Similarity
How similar are consecutive speeches in general?

In [10]:
plt.figure(figsize=(10, 5))
sns.histplot(df['cosine_similarity'], bins=100, kde=True, color='teal')
plt.title('Distribution of Cosine Similarity between Consecutive Speeches')
plt.xlabel('Cosine Similarity Score')
plt.ylabel('Frequency')
plt.show()

print('Summary Statistics:')
display(df['cosine_similarity'].describe())

Summary Statistics:


count    30258.000000
mean         0.667742
std          0.119300
min          0.179132
25%          0.583991
50%          0.671172
75%          0.754508
max          1.000000
Name: cosine_similarity, dtype: float64

## 3. The Most Semantically Similar Conversations
These pairs of speeches have the highest semantic overlap. This often happens in repetitive dialogue, shared metaphors, or when characters echo each other's words.

In [11]:
# Display top 10 most similar interactions
top_similar = df.sort_values('cosine_similarity', ascending=False).head(10)

for i, (idx, row) in enumerate(top_similar.iterrows()):
    print(f"--- Rank {i+1} | Score: {row['cosine_similarity']:.4f} ---")
    print(f"Play: {row['play']} | Scene: {row['scene']}")
    print(f"[{row['speaker1']}]: {row['text1']}")
    print(f"[{row['speaker2']}]: {row['text2']}")
    print('\n')

--- Rank 1 | Score: 1.0000 ---
Play: As You Like It | Scene: A5.S2
[Silvius_AYL]: If this be so, why blame you me to love you?
[Orlando_AYL]: If this be so, why blame you me to love you?


--- Rank 2 | Score: 1.0000 ---
Play: Timon of Athens | Scene: A3.S4
[SERVANTS.VARRO.1_Tim]: My lord—
[SERVANTS.VARRO.2_Tim]: My lord—


--- Rank 3 | Score: 1.0000 ---
Play: Troilus and Cressida | Scene: A4.S2
[Troilus_Tro]: Amen.
[Cressida_Tro]: Amen.


--- Rank 4 | Score: 1.0000 ---
Play: As You Like It | Scene: A5.S2
[Phoebe_AYL]: If this be so, why blame you me to love you?
[Silvius_AYL]: If this be so, why blame you me to love you?


--- Rank 5 | Score: 1.0000 ---
Play: Macbeth | Scene: A4.S1
[WITCHES.2_Mac]: Show.
[WITCHES.3_Mac]: Show.


--- Rank 6 | Score: 1.0000 ---
Play: Macbeth | Scene: A4.S1
[WITCHES.1_Mac]: Show.
[WITCHES.2_Mac]: Show.


--- Rank 7 | Score: 1.0000 ---
Play: Julius Caesar | Scene: A4.S3
[SOLDIERS.BRUTUS.Varro_JC]: My lord?
[SOLDIERS.BRUTUS.Claudius_JC]: My lord?


--- Rank

## 4. The Most Semantically Distinct Transitions
These pairs show the greatest semantic shift between speakers. This might indicate abrupt topic changes, misunderstandings, or formal shifts.

In [12]:
# Display top 10 least similar interactions
bottom_similar = df.sort_values('cosine_similarity', ascending=True).head(10)

for i, (idx, row) in enumerate(bottom_similar.iterrows()):
    print(f"--- Rank {i+1} | Score: {row['cosine_similarity']:.4f} ---")
    print(f"Play: {row['play']} | Scene: {row['scene']}")
    print(f"[{row['speaker1']}]: {row['text1']}")
    print(f"[{row['speaker2']}]: {row['text2']}")
    print('\n')

--- Rank 1 | Score: 0.1791 ---
Play: Richard III | Scene: A1.S3
[QueenMargaret_1H6]: And leave out thee? Stay, dog, for thou shalt hear
me.
If heaven have any grievous plague in store
Exceeding those that I can wish upon thee,
O, let them keep it till thy sins be ripe
And then hurl down their indignation
On thee, the troubler of the poor world’s peace.
The worm of conscience still begnaw thy soul.
Thy friends suspect for traitors while thou liv’st,
And take deep traitors for thy dearest friends.
No sleep close up that deadly eye of thine,
Unless it be while some tormenting dream
Affrights thee with a hell of ugly devils.
Thou elvish-marked, abortive, rooting hog,
Thou that wast sealed in thy nativity
The slave of nature and the son of hell,
Thou slander of thy heavy mother’s womb,
Thou loathèd issue of thy father’s loins,
Thou rag of honor, thou detested—
[RichardIII_R3]: Margaret.


--- Rank 2 | Score: 0.2124 ---
Play: The Taming of the Shrew | Scene: A5.S1
[SERVANTS.PETRUCHIO.Peter_S

## 5. Genre-based Comparison
Does dialogue in Tragedies tend to be more or less semantically coherent than in Comedies?

In [13]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='genre', y='cosine_similarity', data=df, palette='viridis')
plt.title('Speech Similarity by Genre')
plt.show()

print('Mean Similarity by Genre:')
display(df.groupby('genre')['cosine_similarity'].mean().sort_values(ascending=False))

Mean Similarity by Genre:


/tmp/ipykernel_2064006/99114242.py:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='genre', y='cosine_similarity', data=df, palette='viridis')


genre
history    0.688873
comedy     0.665163
tragedy    0.650417
Name: cosine_similarity, dtype: float64

## 6. Chronological Trend
Does Shakespeare's dialogue semantic coherence change over time?

In [14]:
play_avg = df.groupby(['play', 'year', 'genre'])['cosine_similarity'].mean().reset_index()

plt.figure(figsize=(12, 6))
sns.regplot(x='year', y='cosine_similarity', data=play_avg, scatter_kws={'alpha':0.5}, line_kws={'color':'red'})
plt.title('Evolution of Speech Similarity over Time')
plt.xlabel('Year of Composition')
plt.ylabel('Average Cosine Similarity')
plt.show()